In [0]:
from pyspark.sql.functions import col, when

df_audit = spark.table("workspace.gold_sales_silver.flattened_audit").alias("a")
df_txn = spark.table("workspace.gold_sales_silver.enriched_transactions").alias("t")

df_filtered = df_audit.filter(
    (col("a.flag_suspicious") == True) |
    (col("a.flag_manual_review") == True)
)
df_joined = df_filtered.join(
    df_txn,
    col("a.transaction_id") == col("t.transaction_id"),
    "left"
)
df_final = df_joined.select(
    col("a.transaction_id"),
    col("t.branch_id"),
    col("t.customer_segment"),
    col("t.sale_price_usd"),
    col("a.flag_suspicious"),
    col("a.flag_manual_review"),
    when(col("a.flag_suspicious") == True, "HIGH")
    .when(col("a.flag_manual_review") == True, "MEDIUM")
    .otherwise("LOW")
    .alias("alert_severity")
)

In [0]:
target_table = "workspace.gold_sales_gold.compliance_alerts"

if spark.catalog.tableExists(target_table):
    existing = spark.table(target_table).select("transaction_id").distinct()
    
    df_final = df_final.join(existing, on="transaction_id", how="left_anti")

    df_final.write.format("delta") \
        .mode("append") \
        .saveAsTable(target_table)

else:
    df_final.write.format("delta") \
        .mode("overwrite") \
        .saveAsTable(target_table)